<a href="https://colab.research.google.com/github/szczpanski/dev/blob/main/rnn_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Sobre o projeto:

Se trata de um modelo de aprendizagem suprvisionado de classificação binária envolvendo dados relativos ao Câncer de Mama. Os dados foram extraídos do site do Kaggle.


---


Convenções de reprodutibilidade:

Todas as bibliotecas se encontram no arquivo 📄requirements.txt

Para mais informções, consulte nosso README

## Parte 01. Importar os pacotes



---



In [ ]:
!pip install scikeras

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import warnings
warnings.filterwarnings("ignore")

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, roc_curve, auc,
    precision_recall_curve, average_precision_score, classification_report
)
from sklearn.dummy import DummyClassifier

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras import regularizers
from tensorflow.keras.metrics import AUC

from scikeras.wrappers import KerasClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import GridSearchCV
from scipy.stats import uniform, randint


SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

## Parte 02. Baixar e ler os dados

---



O arquivo será baixado diretamente do repositório do Kaggle. Caso a estrutura de pastas não exista, o algoritmo irá construí-la.

In [ ]:
!pip install -q kaggle
!rm -rf kaggle.json
from google.colab import files

files.upload()


In [ ]:
!rm -rf .kaggle
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download --force -d wasiqaliyasir/breast-cancer-dataset
!unzip -o breast-cancer-dataset.zip

In [ ]:
# "breast_cancer.csv"
df = pd.read_csv("Breast_cancer_dataset.csv")

## Parte 03. Análise exploratória


---





In [ ]:
print(f"O dataset possui {df.shape[0]} linhas e {df.shape[1]} colunas.")

3.1. Visão geral do dataset

In [ ]:
def check(df):
    l = []
    colunas = df.columns
    for col in colunas:
        dtypes = df[col].dtypes
        nunique = df[col].nunique()
        sum_null = df[col].isnull().sum()

        moda = df[col].mode().iloc[0] if not df[col].mode().empty else "Não se aplica"
        moda_freq = df[col].value_counts().iloc[0] if not df[col].value_counts().empty else "Não se aplica"

        if np.issubdtype(dtypes, np.number):
            status = df.describe(include='all').T
            media = status.loc[col, 'mean']
            std = status.loc[col, 'std']
            min_val = status.loc[col, 'min']
            quar1 = status.loc[col, '25%']
            mediana = df[col].median()
            quar3 = status.loc[col, '75%']
            max_val = status.loc[col, 'max']
        else:
            status = "Não se aplica"
            media = "Não se aplica"
            std = "Não se aplica"
            min_val = "Não se aplica"
            quar1 = "Não se aplica"
            mediana = "Não se aplica"
            quar3 = "Não se aplica"
            max_val = "Não se aplica"

        l.append([col, dtypes, nunique, sum_null, media, std, min_val, quar1, mediana, quar3, max_val, moda, moda_freq])

    df_check = pd.DataFrame(l, columns=[
        'coluna','tipo','únicos','null_soma','media','desvio',
        'minimo','25%','mediana','75%','maximo','moda','frequência_moda'
    ])
    return df_check

In [ ]:
# Análise geral dos dados
check(df)

In [ ]:
# Análise dos tipos das colunas
df.info()

É possível verificar que o dataset original possui 33 colunas, das quais uma ('Unamed: 32') possui apenas dados nulos.

Além disso, com execeção da coluna 'diagnosis' (que é uma coluna categórica), as demais colunas são colunas numéricas, com informações sobre cada paciente.

### 3.2. Análise da distribuição dos dados

### 3.3. Pairplot - Análise de Relacionamentos entre Variáveis

O **Pairplot** (ou scatterplot matrix) é uma ferramenta fundamental na análise exploratória de dados (EDA). Este gráfico mostra, para cada par de variáveis numéricas, como elas se relacionam, além da distribuição individual em forma de histograma/densidade na diagonal.

**Para que serve**:

* **Exploração de dados (EDA)**: Antes de treinar uma rede neural (ou qualquer modelo de machine learning), é essencial entender como os dados estão distribuídos, se existem correlações fortes entre variáveis e se há separabilidade entre classes.

* **Redução de dimensionalidade** / seleção de features: O pairplot ajuda a identificar variáveis altamente correlacionadas (como radius_mean, perimeter_mean e area_mean), que podem ser redundantes. Isso é útil porque redes neurais podem sofrer com dados redundantes ou multicolinearidade.

* **Identificação de padrões de separação**: Você pode ver se as classes se separam visualmente em determinados pares de features. Isso dá uma intuição de quais variáveis carregam mais poder discriminativo.

**Quando usar em relação a uma rede neural**:

* **Antes do treinamento**: Para inspecionar os dados, escolher features relevantes e entender possíveis ajustes de pré-processamento (normalização, remoção de redundâncias).

* **Não durante o treinamento**: O pairplot é puramente exploratório; não entra como input em uma rede neural. A rede usará os valores numéricos das features (normalizados ou padronizados), não o gráfico em si.

**Fluxo típico**:

1. EDA com pairplot → 2. Pré-processamento (scaling, seleção de features, balanceamento de classes) →
2. Treino da rede neural (ou outro modelo).

Este gráfico é como o 'raio-X' inicial dos dados, antes de colocar a rede para aprender.

In [ ]:
# EDA
_ = check(df)

df_pairplot = df.copy()
label_encoder = LabelEncoder()
df_pairplot['diagnosis_encoded'] = label_encoder.fit_transform(df_pairplot['diagnosis'])

features_for_pairplot = ['radius_mean','texture_mean','perimeter_mean','area_mean','smoothness_mean','diagnosis_encoded']
df_subset = df_pairplot[features_for_pairplot]

print(f"Variáveis selecionadas para o Pairplot: {features_for_pairplot[:-1]}")
print(f"Classes: 0 = Benigno ({sum(df_pairplot['diagnosis_encoded']==0)} casos), 1 = Maligno ({sum(df_pairplot['diagnosis_encoded']==1)} casos)")

In [ ]:
# Configurar o estilo do seaborn
plt.style.use('default')
sns.set_palette("husl")

# Criar o pairplot
fig = plt.figure(figsize=(15, 12))

# Pairplot com separação por classe
pairplot = sns.pairplot(
    df_subset,
    hue='diagnosis_encoded',
    diag_kind='hist',
    plot_kws={'alpha': 0.6, 's': 30},
    diag_kws={'alpha': 0.7}
)

# Personalizar o gráfico
pairplot.fig.suptitle('Pairplot - Análise de Relacionamentos entre Variáveis do Câncer de Mama',
                      fontsize=16, y=1.02)

# Ajustar as legendas
handles = pairplot._legend_data.values()
labels = ['Benigno (0)', 'Maligno (1)']
pairplot.fig.legend(handles, labels, loc='upper right', bbox_to_anchor=(0.98, 0.98))

# Remover a legenda original
pairplot._legend.remove()

plt.tight_layout()
plt.show()

**Análise do Pairplot**:

1. **Correlações Fortes**: Observamos correlações muito fortes entre radius_mean, perimeter_mean e area_mean, o que é esperado geometricamente (raio, perímetro e área estão matematicamente relacionados).

2. **Separabilidade das Classes**:

* Tumores malignos (classe 1) tendem a ter valores maiores de raio, perímetro e área
* Existe uma boa separação visual entre as classes em várias combinações de variáveis
* A textura também mostra alguma capacidade discriminativa

3. **Distribuições**:

* As distribuições na diagonal mostram que algumas variáveis podem se beneficiar de normalização
* Há evidência de que as classes têm distribuições diferentes para a maioria das variáveis

4. **Implicações para a Rede Neural**:

* A forte correlação entre algumas variáveis sugere que podemos considerar redução de dimensionalidade
* A boa separabilidade visual indica que uma rede neural deve conseguir aprender padrões discriminativos
* A necessidade de normalização é evidente devido às diferentes escalas das variáveis.

É possível verificar que o dataset original possui 33 colunas, das quais uma ('Unamed: 32') possui apenas dados nulos.

Além disso, com execeção da coluna 'diagnosis' (que é uma coluna categórica), as demais colunas são colunas numéricas, com informações sobre cada paciente.

### 3.2. Análise da distribuição dos dados

In [ ]:
# Análise da distribuição da variável target
print(f"Distribuição da variável target:")

print(df['diagnosis'].value_counts())

print(f"\nPercentual da distribuição da variável target:")

print(df['diagnosis'].value_counts(normalize=True) * 100)

In [ ]:
# Visualização da distribuição da variável target
plt.figure(figsize=(8, 6))
sns.countplot(data=df, x='diagnosis', palette='viridis')
plt.title('Distribuição da Variável Target (Diagnosis)')
plt.xlabel('Diagnóstico')
plt.ylabel('Frequência')
plt.show()

### Parte 04. Tratamento dos dados (exclusão da coluna nula)

---



In [ ]:
# Removendo a coluna com valores nulos
df = df.drop('Unnamed: 32', axis=1)
print(f"Novo shape do dataset: {df.shape}")

### Parte 05. Verificando a correlação entre as colunas

---



In [ ]:
# Codificando a variável target para análise de correlação
le = LabelEncoder()
df['diagnosis_encoded'] = le.fit_transform(df['diagnosis'])
print(f"Mapeamento: {dict(zip(le.classes_, le.transform(le.classes_)))}")

In [ ]:
# Matriz de correlação
plt.figure(figsize=(20, 16))
correlation_matrix = df.select_dtypes(include=[np.number]).corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('Matriz de Correlação das Variáveis Numéricas')
plt.tight_layout()
plt.show()

### Parte 06. Separar as features utilizadas

---



In [ ]:
# Separando features e target
X = df.drop(['id', 'diagnosis', 'diagnosis_encoded'], axis=1)
y = df['diagnosis_encoded']

print(f"Shape das features (X): {X.shape}")
print(f"Shape do target (y): {y.shape}")

### Parte 07. Normalização com o Standard Scaler

---



In [ ]:
# Divisão treino/teste
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)
print(f"Shape X_train: {X_train.shape}")
print(f"Shape X_test: {X_test.shape}")
print(f"Shape y_train: {y_train.shape}")
print(f"Shape y_test: {y_test.shape}")

In [ ]:
# Normalização dos dados
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f"Média antes da normalização: {X_train.mean().mean():.4f}")
print(f"Desvio padrão antes: {X_train.std().mean():.4f}")
print(f"Média após normalização: {X_train_scaled.mean():.4f}")
print(f"Desvio padrão após: {X_train_scaled.std():.4f}")

### Parte 08. Modelos baseline

---



8.1. Criando um dumb model (baseline) prevendo tudo como a classe majoritária para validações futuras

In [ ]:
# Baseline trivial (Dummy)
dummy = DummyClassifier(strategy='most_frequent', random_state=SEED)
dummy.fit(X_train_scaled, y_train)
y_pred_dummy = dummy.predict(X_test_scaled)

print("=== DUMB MODEL (Baseline) ===")
print(f"Accuracy:  {accuracy_score(y_test, y_pred_dummy):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_dummy, zero_division=0):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_dummy, zero_division=0):.4f}")
print(f"F1-Score:  {f1_score(y_test, y_pred_dummy, zero_division=0):.4f}")

8.2. Criando um baseline de Keras

In [ ]:
# Modelo baseline com Keras
def create_baseline_model():
    model = Sequential([
        Dense(64, activation='relu', input_shape=(X_train_scaled.shape[1],)),
        Dense(32, activation='relu'),
        Dense(1, activation='sigmoid')
    ])

    model.compile(optimizer='adam',
                  loss='binary_crossentropy',
                  metrics=['accuracy'])

    return model

# Criando e treinando o modelo
baseline_model = create_baseline_model()

# EarlyStopping
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

history_baseline = baseline_model.fit(X_train_scaled, y_train,
                                      validation_split=0.2,
                                      epochs=100,
                                      batch_size=32,
                                      verbose=1,
                                      callbacks=[early_stop])

# Predições
y_pred_baseline_prob = baseline_model.predict(X_test_scaled)
y_pred_baseline = (y_pred_baseline_prob > 0.5).astype(int).flatten()

# Métricas
print(f"Accuracy:  {accuracy_score(y_test, y_pred_baseline):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_baseline):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_baseline):.4f}")
print(f"F1-Score:  {f1_score(y_test, y_pred_baseline):.4f}")

### Parte 09. Grid search

---



In [ ]:
def create_model(neurons1=16, neurons2=8, dropout_rate=0.3, l2_lambda=0.001):
    model = Sequential([
        Dense(neurons1, activation='relu', kernel_regularizer=regularizers.l2(l2_lambda), input_shape=(X_train_scaled.shape[1],)),
        Dropout(dropout_rate),
        Dense(neurons2, activation='relu', kernel_regularizer=regularizers.l2(l2_lambda)),
        Dropout(dropout_rate),
        Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['recall', 'accuracy'])
    return model

# Define o callback de EarlyStopping
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

# Configuração do Grid Search
keras_classifier = KerasClassifier(
    model=create_model,
    epochs=100,
    batch_size=32,
    verbose=0,
    # Adicionando o EarlyStopping como um callback
    callbacks=[early_stop]
)

# Grade de parâmetros
params_grid = {
    'model__optimizer': ['adam', 'rmsprop'],
    'model__learning_rate': [0.001, 0.01],
    'model__hidden_layers': [[n] for n in range(2, 11)],
    'model__dropout_rate': [0.2, 0.3, 0.4],
    'model__l2_lambda': [0.001, 0.01, 0.1]
}

# Validação cruzada com 5 folds
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    estimator=keras_classifier,
    param_grid=params_grid,
    scoring='accuracy',
    cv=cv_strategy,
    n_jobs=-1,
    verbose=1
)

grid_search_result = grid_search.fit(X_train_scaled, y_train)

# Melhores parâmetros e melhor pontuação
print("Melhores parâmetros encontrados: ", grid_search_result.best_params_)
print("Melhor pontuação (acurácia): %.4f" % grid_search_result.best_score_)

# Treinar o modelo com os melhores parâmetros encontrados
best_classifier = grid_search_result.best_estimator_
best_classifier.fit(X_train_scaled, y_train)
y_pred_best = best_classifier.predict(X_test)

### Parte 10. Comparação dos modelos

---



In [ ]:
# Gráfico 1: Histórico de Treinamento do MELHOR Modelo
if best_history:
    plt.style.use('seaborn-v0_8-whitegrid')
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

    # Gráfico de Recall
    ax1.plot(best_history.history['recall'], label='Recall de Treino', color='royalblue')
    ax1.plot(best_history.history['val_recall'], label='Recall de Validação', color='darkorange')
    ax1.set_title('Histórico de Recall do Melhor Modelo')
    ax1.set_xlabel('Época')
    ax1.set_ylabel('Recall')
    ax1.legend()

    # Gráfico de Perda (Loss)
    ax2.plot(best_history.history['loss'], label='Perda de Treino', color='royalblue')
    ax2.plot(best_history.history['val_loss'], label='Perda de Validação', color='darkorange')
    ax2.set_title('Histórico de Perda (Loss) do Melhor Modelo')
    ax2.set_xlabel('Época')
    ax2.set_ylabel('Loss')
    ax2.legend()

    plt.suptitle(f"Análise de Treinamento do Modelo {best_config}", fontsize=16)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()

# Gráfico 2: Comparação de F1-Score Final Entre Todos os Modelos
if results:
    sorted_results = sorted(results.items(), key=lambda item: item[1]['f1'], reverse=True)
    configs = [item[0] for item in sorted_results]
    f1_scores = [item[1]['f1'] for item in sorted_results]

    plt.figure(figsize=(10, 6))
    bars = plt.bar(configs, f1_scores, color='skyblue', edgecolor='black')
    plt.xlabel('Configuração de Neurônios (Camada 1, Camada 2)')
    plt.ylabel('F1-Score Final no Conjunto de Teste')
    plt.title('Comparação de F1-Score entre Arquiteturas')
    plt.xticks(rotation=45, ha="right")
    plt.ylim(0, max(f1_scores) * 1.1)

    for bar in bars:
        yval = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2.0, yval, f'{yval:.3f}', va='bottom', ha='center')

    plt.tight_layout()
    plt.show()

In [ ]:
models_comparison = pd.DataFrame({
    'Modelo': ['Dumb Classifier', 'Keras Baseline', 'Grid Search'],
    'Accuracy': [
        accuracy_score(y_test, y_pred_dummy),
        accuracy_score(y_test, y_pred_baseline),
        accuracy_score(y_test, y_pred_best)
    ],
    'Precision': [
        precision_score(y_test, y_pred_dummy, zero_division=0),
        precision_score(y_test, y_pred_baseline),
        precision_score(y_test, y_pred_best)
    ],
    'Recall': [
        recall_score(y_test, y_pred_dummy, zero_division=0),
        recall_score(y_test, y_pred_baseline),
        recall_score(y_test, y_pred_best)
    ],
    'F1-Score': [
        f1_score(y_test, y_pred_dummy, zero_division=0),
        f1_score(y_test, y_pred_baseline),
        f1_score(y_test, y_pred_best)
    ]
})

print("=== COMPARAÇÃO DOS MODELOS ===")
print(models_comparison.round(4))


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

models_preds = [y_pred_dummy, y_pred_baseline, y_pred_best]
model_names = ['Dumb Classifier', 'Keras Baseline', 'Grid search']

for i, (pred, name) in enumerate(zip(models_preds, model_names)):
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i])
    axes[i].set_title(f'Matriz de Confusão - {name}')
    axes[i].set_xlabel('Predito')
    axes[i].set_ylabel('Real')

plt.tight_layout()
plt.show()
